# Network Expansion
## Model 2 (Test) - Intertemporal Deterministic Model

True multi-year formulation solved in one MILP. Investments are time-indexed (years), activations are non-decreasing, and capacity reinforcements accumulate. Costs are discounted and budgets enforced per year.

### 1 - Imports

In [7]:
import numpy as np
import pandas as pd

from src.classes import DistributionNetwork, Substation
from src.solver import solve_network_intertemporal

### 2 - Define the Distribution Network (shared across years)

In [8]:
# Nodes and loads
NODES = [f"N{i}" for i in range(1, 14)]
LOADS = [f"D{i}" for i in range(1, 11)]

# Initial substation
S1 = Substation("S1", "N4", 40, ['N3', 'N5', 'N9'], r_cost=200, edge_cost=50)
SUBSTATIONS = [S1]

line_cost = 50

base_load_capacity = {
    'D1': 6, 'D2': 3, 'D3': 2, 'D4': 5, 'D5': 3,
    'D6': 2, 'D7': 3, 'D8': 5, 'D9': 4, 'D10': 6
}

loads_locations = {
    'D1': 'N1', 'D2': 'N2', 'D3': 'N3', 'D4': 'N6', 'D5': 'N7',
    'D6': 'N8', 'D7': 'N9', 'D8': 'N11', 'D9': 'N12', 'D10': 'N13'
}

nodes_connected = {
    'N1': ['N2'],
    'N2': ['N1', 'N3'],
    'N3': ['N2', 'N4'],
    'N4': ['N3', 'N5', 'N9'],
    'N5': ['N4', 'N6'],
    'N6': ['N5','N7', 'N8'],
    'N7': ['N6'],
    'N8': ['N6'],
    'N9': ['N4', 'N10'],
    'N10': ['N9', 'N11', 'N13'],
    'N11': ['N10', 'N12'],
    'N12': ['N11'],
    'N13': ['N10']
}

DistributionNetwork = DistributionNetwork(
    NODES.copy(),
    LOADS.copy(),
    SUBSTATIONS.copy(),
    base_load_capacity.copy(),
    nodes_connected.copy(),
    loads_locations.copy(),
    line_cost
)

# Candidate substations (same as Model 1)
capacity = 15
s_cost = 100       # activation cost
l_cost = line_cost # feeder line cost
r_cost = 200       # capacity reinforcement cost

S2 = Substation("S2", "N14", capacity, ['N2'], r_cost, edge_cost=l_cost, fix_cost=s_cost)
S3 = Substation("S3", "N15", capacity, ['N6'], r_cost, edge_cost=l_cost, fix_cost=s_cost)
S4 = Substation("S4", "N16", capacity, ['N11', 'N13'], r_cost, edge_cost=l_cost, fix_cost=s_cost)

DistributionNetwork.add_candidate_substations([S2, S3, S4])

### 3 - Multi-year data and scenario (deterministic growth)

In [9]:
years = list(range(1, 11))      # Years 1..10
demand_rate = 0.06                 # Annual growth (deterministic)
R = 10                             # Size of capacity reinforcement
# Dynamic annual budgets: starts at minimum 130 and grows by 25 per year
B = {t: 1300 + 25*(t-1) for t in years}
dr = 0.05                          # Discount rate

# Build per-year demand dictionaries
demands = {}
for t in years:
    factor = (1 + demand_rate) ** (t - 1)
    demands[t] = {ld: val * factor for ld, val in base_load_capacity.items()}

# Quick check of total demand per year
system_demand = {t: round(sum(demands[t].values()), 2) for t in years}
system_demand

{1: 39.0,
 2: 41.34,
 3: 43.82,
 4: 46.45,
 5: 49.24,
 6: 52.19,
 7: 55.32,
 8: 58.64,
 9: 62.16,
 10: 65.89}

### 4 - Solve intertemporal model

In [10]:
solution = solve_network_intertemporal(
    DistributionNetwork,
    R=R,
    B=B,
    dr=dr,
    years=years,
    demands=demands,
    OutputFlag=0
)

### 5 - Summaries

In [11]:
S_idx = list(range(1, len(DistributionNetwork.SUBSTATIONS)+1))
base_edge_cost = DistributionNetwork.edge_cost

# Helper: list NEW lines built per year (costed edges only)
lines_added = {}
for t in years:
    added = set()
    for (i, j, s, tau), val in solution['b_on'].items():
        if tau == t and val > 0.5:
            e = (i, j)
            # include only edges with positive base cost (new builds)
            if base_edge_cost.get(e, 0) > 0:
                added.add(e)
    lines_added[t] = sorted(list(added))

summary = pd.DataFrame({
    'Budget': [B[t] if not isinstance(B, dict) else B[t] for t in years],
    'System Demand': [system_demand[t] for t in years],
    'System Supply': [round(sum(solution['r'][(s, t)] for s in S_idx), 2) for t in years],
    'System Capacity': [sum(solution['P'][(s, t)] for s in S_idx) for t in years],
    'Substations Active': [[s for s in S_idx if solution['w'][(s, t)] > 0.5] for t in years],
    'Lines Added': [lines_added[t] for t in years],
    'Cost substation (nom)': [round(solution['cost_components_nominal'][t]['substation'], 2) for t in years],
    'Cost lines (nom)': [round(solution['cost_components_nominal'][t]['lines'], 2) for t in years],
    'Cost reinforcement (nom)': [round(solution['cost_components_nominal'][t]['reinforcement'], 2) for t in years],
    'Cost (nominal)': [round(solution['cost_per_year_nominal'][t], 2) for t in years],
    'Cost (discounted)': [round(solution['cost_per_year'][t], 2) for t in years]
}, index=years)

summary

,Budget,System Demand,System Supply,System Capacity,Substations Active,Lines Added,Cost substation (nom),Cost lines (nom),Cost reinforcement (nom),Cost (nominal),Cost (discounted)
1,1300,39.00,39.00,40.0,[1],[],0.0,0.0,0.0,0.0,0.00
2,1325,41.34,41.34,55.0,"[1, 2]","[(2, 14)]",100.0,50.0,0.0,150.0,142.86
3,1350,43.82,43.82,55.0,"[1, 2]",[],0.0,0.0,0.0,0.0,0.00
4,1375,46.45,46.45,55.0,"[1, 2]",[],0.0,0.0,0.0,0.0,0.00
5,1400,49.24,49.24,55.0,"[1, 2]",[],0.0,0.0,0.0,0.0,0.00
6,1425,52.19,52.19,55.0,"[1, 2]",[],0.0,0.0,0.0,0.0,0.00
7,1450,55.32,55.32,70.0,"[1, 2, 4]","[(13, 16)]",100.0,50.0,0.0,150.0,111.93
8,1475,58.64,58.64,70.0,"[1, 2, 4]",[],0.0,0.0,0.0,0.0,0.00
9,1500,62.16,62.16,70.0,"[1, 2, 4]",[],0.0,0.0,0.0,0.0,0.00
10,1525,65.89,65.89,80.0,"[1, 2, 4]",[],0.0,0.0,200.0,200.0,128.92


### 6 - Final year assignments

In [12]:
last_year = years[-1]
S = list(np.arange(1, len(DistributionNetwork.SUBSTATIONS)+1))
N = list(np.arange(1, len(DistributionNetwork.NODES)+1))
print(f"Node assignments in Year {last_year}:")
for n in N:
    for s in S:
        if solution['y'][(n, s, last_year)] > 0.5:
            print(f"Node {DistributionNetwork.NODES[n-1]} assigned to {DistributionNetwork.SUBSTATIONS[s-1].id}")


Node assignments in Year 10:
Node N1 assigned to S2
Node N2 assigned to S2
Node N3 assigned to S2
Node N4 assigned to S1
Node N5 assigned to S1
Node N6 assigned to S1
Node N7 assigned to S1
Node N8 assigned to S1
Node N9 assigned to S1
Node N10 assigned to S1
Node N11 assigned to S1
Node N12 assigned to S1
Node N13 assigned to S4
Node N14 assigned to S2
Node N16 assigned to S4
